In [1]:
# import json
import os

import dotenv
import requests
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import (  # FunctionTool,; RequiredFunctionToolCall,
    ListSortOrder,
    MessageRole,
    SubmitToolOutputsAction,
    ToolOutput,
)
from azure.identity import DefaultAzureCredential

In [2]:
dotenv.load_dotenv()

BALANCE_AGENT_ID = os.environ["BALANCE_AGENT_ID"]
PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
BALANCE_API_BASE_URL = os.environ.get("BALANCE_API_BASE_URL", "http://localhost:8000")

In [3]:
client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=DefaultAzureCredential(),
)

In [4]:
# def call_agent(user_input: str) -> str:
#     thread = client.threads.create()
#     # print('thread id:', thread.id)

#     client.messages.create(
#         thread_id=thread.id,
#         role=MessageRole.USER,
#         content=user_input,
#     )

#     client.runs.create_and_process(
#         thread_id=thread.id,
#         agent_id=BALANCE_AGENT_ID,
#     )

#     messages = client.messages.list(
#         thread_id=thread.id,
#         order=ListSortOrder.ASCENDING,
#     )

#     return messages

In [5]:
def get_current_balance(customer_id: str) -> str:
    base_url = BALANCE_API_BASE_URL.rstrip("/")
    url = f"{base_url}/api/balance"
    print("get_current_balance() called with:", customer_id, "->", url)

    response = requests.post(
        url,
        json={"customer_id": customer_id},
        timeout=5,
    )
    print("mock status_code:", response.status_code, "body:", response.text)
    response.raise_for_status()
    return response.text


def call_agent(user_input: str):
    client = AgentsClient(
        endpoint=PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )

    with client:
        thread = client.threads.create()

        client.messages.create(
            thread_id=thread.id,
            role=MessageRole.USER,
            content=user_input,
        )

        run = client.runs.create(
            thread_id=thread.id,
            agent_id=BALANCE_AGENT_ID,
        )

        while True:
            run = client.runs.get(
                thread_id=thread.id,
                run_id=run.id,
            )
            print("RUN STATUS:", run.status)

            if run.status == "requires_action":
                action = run.required_action

                if isinstance(action, SubmitToolOutputsAction):
                    tool_calls = action.submit_tool_outputs.tool_calls
                else:
                    print(
                        "Required action no es SubmitToolOutputsAction:", type(action)
                    )
                    break

                tool_outputs: list[ToolOutput] = []

                for tool_call in tool_calls:
                    print("TOOL CALL:", tool_call, type(tool_call))
                    output = functions.execute(tool_call)
                    print("TOOL OUTPUT (raw):", output)
                    tool_outputs.append(
                        ToolOutput(
                            tool_call_id=tool_call.id,
                            output=output,
                        )
                    )

                run = client.runs.submit_tool_outputs(
                    thread_id=thread.id,
                    run_id=run.id,
                    tool_outputs=tool_outputs,
                )
                continue

            if run.status in ("completed", "failed", "cancelled"):
                break

        messages = client.messages.list(
            thread_id=thread.id,
            order=ListSortOrder.ASCENDING,
        )

        return list(messages)

In [6]:
get_current_balance('customer-001')

get_current_balance() called with: customer-001 -> https://microsoft-foundry-playground-balance-mock-api.9uxguk.easypanel.host/api/balance
mock status_code: 200 body: {
  "balance_actual": 152000.5
}



'{\n  "balance_actual": 152000.5\n}\n'

In [7]:
# messages = call_agent("Super califragilistico espialidoso")
messages = call_agent(
    "Hola. Mi customer_id es customer-002. ¿Cuál es el saldo actual de mi cuenta?"
)

for msg in messages:
    print(f"{msg.role}:")
    for text_msg in msg.text_messages:
        print(text_msg.text.value)
    print()

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: Failed to invoke PowerShell
	AzureDeveloperCliCredential: {"type":"consoleMessage","timestamp":"2026-04-07T17:41:18.2279754-04:00","data":{"message":"\nERROR: fetching token: AADSTS700082: The refresh token has expired due to inactivity.Â The token was issued on 2025-11-27T00:25:58.3203582Z and was inactive for 90.00:00:00. Trace ID: a85ccdb3-0b41-4c

ClientAuthenticationError: DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	AzureCliCredential: Azure CLI not found on path
	AzurePowerShellCredential: Failed to invoke PowerShell
	AzureDeveloperCliCredential: {"type":"consoleMessage","timestamp":"2026-04-07T17:41:18.2279754-04:00","data":{"message":"\nERROR: fetching token: AADSTS700082: The refresh token has expired due to inactivity.Â The token was issued on 2025-11-27T00:25:58.3203582Z and was inactive for 90.00:00:00.